# Global eCommerce Performance Analysis
### Data-Driven Sales & Logistics Insights

**Project Overview:**

This project provides a comprehensive exploratory data analysis (EDA) of global retail operations. By leveraging Python and Interactive Plotly visualizations, it uncovers key trends in international revenue, optimizes logistics by identifying shipping outliers, and segments customer behavior to drive strategic business growth.

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import plotly.io as pio

# This tells Kaggle to render the charts in a way that viewers can see
pio.renderers.default = 'iframe'

#Sleep Warnings
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Set Styling
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize']

# Load dataset
df = pd.read_csv('./dataset/global_ecommerce_sales.csv')

# Chek the shape of the dataset
print(f"Shape of the dataset: \n {df.shape}")

print('=' * 70)

# Columns names of dataset
print(f"Column Names: \n {df.columns}")

Shape of the dataset: 
 (2000, 15)
Column Names: 
 Index(['Order_ID', 'Order_Date', 'Customer_Name', 'Customer_Segment',
       'Country', 'Region', 'Product_Category', 'Product_Name', 'Quantity',
       'Unit_Price', 'Discount_Percent', 'Total_Sales', 'Shipping_Cost',
       'Profit', 'Payment_Method'],
      dtype='object')


In [76]:
# Check the first five rows of the dataset
df.head() 

,Order_ID,Order_Date,Customer_Name,Customer_Segment,Country,Region,Product_Category,Product_Name,Quantity,Unit_Price,Discount_Percent,Total_Sales,Shipping_Cost,Profit,Payment_Method
0,ORD-11121,2023-01-02,Karen Suzuki,Corporate,United States,North America,Technology,Wireless Bluetooth Headphones,3,99.43,0,298.29,9.31,124.92,Cash on Delivery
1,ORD-11244,2023-01-02,John Johansson,Corporate,Spain,Europe,Technology,Mechanical Gaming Keyboard,4,97.93,20,313.38,14.31,83.62,Cash on Delivery
2,ORD-10325,2023-01-03,Jessica Garcia,Consumer,Mexico,North America,Office Supplies,Binder Clips Assorted 48pc,2,10.74,0,21.48,8.12,3.69,Credit Card
3,ORD-10467,2023-01-03,Clara Taylor,Corporate,Italy,Europe,Technology,Webcam HD 1080p,2,61.86,15,105.16,10.79,26.32,PayPal
4,ORD-11454,2023-01-05,Felix Thomas,Consumer,Italy,Europe,Furniture,Standing Desk Converter,4,330.67,20,1058.14,11.09,253.44,Credit Card


In [77]:
# Let's Check the info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          2000 non-null   object 
 1   Order_Date        2000 non-null   object 
 2   Customer_Name     2000 non-null   object 
 3   Customer_Segment  2000 non-null   object 
 4   Country           2000 non-null   object 
 5   Region            2000 non-null   object 
 6   Product_Category  2000 non-null   object 
 7   Product_Name      2000 non-null   object 
 8   Quantity          2000 non-null   int64  
 9   Unit_Price        2000 non-null   float64
 10  Discount_Percent  2000 non-null   int64  
 11  Total_Sales       2000 non-null   float64
 12  Shipping_Cost     2000 non-null   float64
 13  Profit            2000 non-null   float64
 14  Payment_Method    2000 non-null   object 
dtypes: float64(4), int64(2), object(9)
memory usage: 234.5+ KB


In [78]:
# Check the over all null values throughout the whole dataset without showing every feature 
df.isnull().sum().sum()

0

In [79]:
# Duplicate Values
df.duplicated().sum()

0

### Regions wise Sales Analysis

In [80]:
# Grouping the Region to see the real winners
region_analysis = df.groupby('Region').agg({
    'Total_Sales': 'sum',
    'Profit': 'sum',
    'Shipping_Cost': 'mean'
}).sort_values(by='Profit', ascending=False)

print(region_analysis)

                      Total_Sales    Profit  Shipping_Cost
Region                                                    
Europe                  137006.20  45672.16      12.304911
North America           133876.38  45250.09      10.046419
Asia Pacific            121707.51  39116.61      14.482981
South America            46051.13  14680.98      15.444241
Middle East & Africa     45918.12  14152.48      15.995385


In [81]:
import plotly.graph_objects as go

# 1. Preparing the data
region_summary = df.groupby('Region').agg({
    'Total_Sales': 'sum',
    'Profit': 'sum'
}).sort_values(by='Total_Sales', ascending=False).reset_index()

# 2. Creating a "Beautiful" Figure
fig = go.Figure()

# Add Sales Bar (Deep Blue)
fig.add_trace(go.Bar(
    x=region_summary['Region'],
    y=region_summary['Total_Sales'],
    name='Total Sales',
    marker_color='#2E86C1',  # Professional Blue
    opacity=0.8,
    text=region_summary['Total_Sales'],
    texttemplate='%{text:.2s}', # Displays in 'k' or 'M' format
    textposition='outside'
))

# Add Profit Bar (Emerald Green)
fig.add_trace(go.Bar(
    x=region_summary['Region'],
    y=region_summary['Profit'],
    name='Profit',
    marker_color='#27AE60',  # Success Green
    opacity=0.9,
    text=region_summary['Profit'],
    texttemplate='%{text:.2s}',
    textposition='outside'
))

# 3. Styling the Layout for Maximum Visual Appeal
fig.update_layout(
    title={
        'text': "<b>🌍 Global Performance: Sales vs. Profit by Region</b>",
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top',
        'font': {'size': 24, 'family': 'Arial Black'}
    },
    xaxis_title="<b>Region</b>",
    yaxis_title="<b>Value (USD)</b>",
    barmode='group',          # Places bars side-by-side
    template='plotly_white',  # Clean white background
    bargap=0.2,               # Gap between bars of different groups
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1,
        bgcolor="rgba(255, 255, 255, 0.5)"
    ),
    margin=dict(l=50, r=50, t=120, b=50),
    hovermode="x unified"      # Shows both values on a single hover popup
)

# Render the plot
fig.show()

### Analysis:

Here are the three high-impact bullet points for your Global Sales vs. Order Volume analysis:

**`Market Concentration:`** High-sales regions like Europe ($137k) and North America ($133.8k) are driven by high order densities, particularly in countries like Mexico (203 orders) and the United States (196 orders).

**`Revenue Efficiency:`** While North America leads in order volume, regions like Europe show higher total sales with slightly fewer orders, suggesting a higher average transaction value per customer.

**`Expansion Opportunities:`** Markets such as Middle East & Africa and South America show lower order volumes (e.g., UAE with only 47 orders), indicating significant untapped potential for scaling order frequency through targeted marketing.

---

## 🛠️ Advanced Feature Engineering & Business Metrics
In this section, we transform raw data into high-value features. By extracting time-based components from Order_Date, we enable Year-over-Year comparisons. Additionally, we calculate Customer Lifetime Value (CLV) and Shipping Efficiency to provide deeper strategic insights into customer loyalty and logistics costs.

In [82]:
# --- Advanced Feature Engineering ---

# 1. Convert Order_Date to datetime objects
df['Order_Date'] = pd.to_datetime(df['Order_Date'])

# 2. Extract Time Features for Seasonality Analysis
df['Order_Month'] = df['Order_Date'].dt.month_name()
df['Order_Year'] = df['Order_Date'].dt.year
df['Month_Year'] = df['Order_Date'].dt.to_period('M').astype(str)

# 3. Calculate Shipping-to-Sales Ratio (Efficiency Metric)
# This identifies where logistics costs are eating into margins
df['Shipping_Impact_Pct'] = (df['Shipping_Cost'] / df['Total_Sales']) * 100


# --- Customer Lifetime Value (CLV) Calculation ---
# PURPOSE: We calculate CLV to segment our user base. 
# While new customers join daily, this metric allows us to distinguish 
# between one-time shoppers and high-value loyal "VIP" customers.
# This helps the business decide where to focus marketing and loyalty rewards.

# 1. Summarizing total historical spend per customer
customer_clv = df.groupby('Customer_Name')['Total_Sales'].sum().reset_index()
customer_clv.columns = ['Customer_Name', 'CLV']

# 2. Merging CLV back to the main data so every row shows the customer's total value
df = df.merge(customer_clv, on='Customer_Name', how='left')

print(f"✅ CLV calculation complete. Identified {df['Customer_Name'].nunique()} unique customers.")

# Show the first few rows to verify the newly created features
print("✅ Feature Engineering Complete: New Features are added.")
new_cols = ['Order_Date', 'Order_Year', 'Order_Month', 'Shipping_Impact_Pct', 'CLV']
df[new_cols].head()


✅ CLV calculation complete. Identified 1534 unique customers.
✅ Feature Engineering Complete: New Features are added.


,Order_Date,Order_Year,Order_Month,Shipping_Impact_Pct,CLV
0,2023-01-02,2023,January,3.121124,972.28
1,2023-01-02,2023,January,4.566341,313.38
2,2023-01-03,2023,January,37.802607,118.78
3,2023-01-03,2023,January,10.260555,105.16
4,2023-01-05,2023,January,1.048065,1058.14


In [83]:
# 1. Prepare the data 
monthly_sales = df.groupby(['Order_Year', 'Order_Month'])['Total_Sales'].sum().reset_index()

# 2. Ensure months are in order
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_sales['Order_Month'] = pd.Categorical(monthly_sales['Order_Month'], categories=month_order, ordered=True)
monthly_sales = monthly_sales.sort_values(['Order_Year', 'Order_Month'])

# 3. Create the Line Chart
fig_trend = px.line(
    monthly_sales, 
    x='Order_Month', 
    y='Total_Sales', 
    color='Order_Year',  # This creates one line per year
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Prism, 
    title="<b>📈 Year-over-Year Sales Momentum</b>",
    labels={'Order_Month': 'Month', 'Total_Sales': 'Revenue (USD)', 'Order_Year': 'Year'},
    template="plotly_white"
)

# 4. Styling
fig_trend.update_layout(
    title_x=0.5,
    hovermode="x unified"
)
fig_trend.update_yaxes(tickprefix="$", tickformat=",.2s") # Adds $ and simplifies numbers (e.g., 2M)

fig_trend.show()

### 📊 Analysis: Monthly Sales Momentum

**`Seasonality Trends:`** Reveals consistent peaks and valleys (e.g., Q4 holiday spikes). This helps the business optimize inventory levels and staffing ahead of high-demand periods.

**`Year-over-Year (YoY) Growth:`** By overlaying multiple years, we can instantly validate if current growth is outpacing historical performance at the same point in time.

**`Strategic Insight:`** This visualization acts as the `"Business Pulse."` It identifies whether sales are accelerating or if specific low-performing months require targeted promotional campaigns to bridge revenue gaps.

---

### Country Wise Sales Analysis

In [84]:
# --- STEP 1: THE FIGURES (Numerical Summary) ---
country_summary = df.groupby('Country').agg({
    'Total_Sales': 'sum',
    'Quantity': 'sum',
    'Profit': 'sum'
}).sort_values('Total_Sales', ascending=False).reset_index()

# Adding Profit Margin for extra professional insight
country_summary['Profit_Margin %'] = (country_summary['Profit'] / country_summary['Total_Sales']) * 100

# Displaying the Top 10 Countries as a styled "Figure"
print("Top 10 Countries by Revenue and Profitability:")
country_summary.head(20).style.background_gradient(cmap='Greens', subset=['Total_Sales', 'Profit'])\
    .format({'Total_Sales': '${:,.2f}', 'Profit': '${:,.2f}', 'Profit_Margin %': '{:.2f}%'})

Top 10 Countries by Revenue and Profitability:


,Country,Total_Sales,Quantity,Profit,Profit_Margin %
0,Mexico,"$47,217.30",712,"$15,949.44",33.78%
1,Canada,"$45,326.55",620,"$15,320.95",33.80%
2,United States,"$41,332.53",664,"$13,979.70",33.82%
3,Japan,"$30,950.19",497,"$9,826.31",31.75%
4,United Kingdom,"$30,185.46",374,"$10,185.32",33.74%
5,Germany,"$28,989.73",399,"$9,518.07",32.83%
6,China,"$28,322.85",409,"$9,028.07",31.88%
7,Italy,"$27,484.38",378,"$9,125.06",33.20%
8,France,"$26,616.81",393,"$8,973.46",33.71%
9,Australia,"$25,287.72",382,"$8,072.37",31.92%


In [85]:
# Show the whole countries name 
df['Country'].value_counts()

Country
Mexico            203
United States     196
Canada            179
Japan             124
China             109
Australia         107
United Kingdom    106
Italy             105
France            105
Germany            95
Spain              92
India              92
South Korea        88
Argentina          66
Brazil             65
South Africa       62
Colombia           60
Saudi Arabia       53
UAE                47
Nigeria            46
Name: count, dtype: int64

**Note:** As there are 20 countries, So we decided to see the total `Sales` & `Profit` of every country. 

In [86]:
# 1. Prepare data with Sales and Count
country_stats = df.groupby('Country').agg({
    'Total_Sales': 'sum',
    'Order_ID': 'count'
}).reset_index().rename(columns={'Order_ID': 'Number_of_Orders'})

# 2. Create the Plot
fig_map = px.choropleth(
    country_stats,
    locations="Country",
    locationmode='country names',
    color="Total_Sales",
    hover_name="Country",
    # This is the secret to a 'clean' look:
    hover_data={
        'Country': False,              # Hide this (it's already in the title)
        'Total_Sales': ':$,.2f',       # Format as currency with 2 decimals
        'Number_of_Orders': True       # Show the count of orders
    },
    color_continuous_scale='Turbo',    # Very vibrant and attractive
    title="<b>Global Sales & Order Volume</b>"
)

fig_map.update_layout(
    title_x=0.5,
    margin={"r":0,"t":80,"l":0,"b":0}
)

fig_map.show()

### Country Wise Analysis:

`Dominant Market Leaders:` Identifies Mexico ($47.2k) and Canada ($45.3k) as the top revenue-generating countries, significantly outperforming other international markets in total sales value.

`Order Density Correlation:` Highlights a direct link between high sales and order volume, with Mexico leading the dataset with 203 orders, followed closely by the United States with 196 orders.

`Geographic Performance Gaps:` The interactive Choropleth Map reveals emerging markets like the UAE and Colombia currently have the lowest order volumes (under 60 orders each), signaling key areas for potential market penetration and expansion.

---

### Customer_Segment vs Product_Category

In [87]:
# 1. Create the Sunburst
# We define the 'path' as the levels of hierarchy we want to see
fig_sun = px.sunburst(
    df, 
    path=['Customer_Segment', 'Product_Category'], 
    values='Total_Sales',
    color='Profit',                      # Use Profit for the color to see which segments are actually making money
    color_continuous_scale='RdYlGn',     # Red (Loss) -> Yellow -> Green (Profit)
    title="<b>Sales Hierarchy: Segment & Category Performance</b>",
    template='plotly_white'
)

# 2. Make it "Beautiful and Attractive"
fig_sun.update_layout(
    title={
        'y': 0.95,
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 24, 'family': 'Arial Black'}
    },
    margin=dict(t=80, l=0, r=0, b=0),
    # Add a nice glow/shadow effect to the paper
    paper_bgcolor='white'
)

# 3. Update the hover information
fig_sun.update_traces(
    hovertemplate='<b>%{label}</b><br>Sales: $%{value:,.2f}<br>Profit: $%{color:,.2f}'
)

fig_sun.show()

**Why the colors are different:**

**Profitability, not Category:** 

Plotly isn't coloring the slices based on what they are (e.g., all "Technology" being blue). Instead, it's coloring them based on how much money they made.

**The Scale (RdYlGn):** We used a Red-Yellow-Green scale.

**`Dark Green:`** High Profit.

**`Yellow:`** Break-even / Low Profit.

**`Red:`** Loss (Negative Profit).

**Note:**

In the above `Pie Chart`, the size of the slice represents the `Sales` and the color of the slice represents the `Profit`

---

### The 10 Most Prifitable Products

In [88]:
# 1. Get the Top 10 by Profit
top_10_products = df.groupby('Product_Name')['Profit'].sum().nlargest(10).reset_index()

# 2. Create a Sleek Horizontal Bar Chart
fig_prod = px.bar(
    top_10_products, 
    x='Profit', 
    y='Product_Name', 
    orientation='h',  # Horizontal
    color='Profit', 
    color_continuous_scale='Greens', # Money colors!
    title="<b>🏆 Top 10 Profit-Generating Products</b>",
    template='plotly_white'
)

# 3. Style it for "Premium" look
fig_prod.update_layout(
    yaxis={'categoryorder':'total ascending'}, # Best at the top
    title_x=0.5,
    margin=dict(l=200) # Give room for long product names
)

fig_prod.show()

### Analysis of Top Products:

**`Profitability Leaders:`** Identifies the "Star" products that generate the highest net income, allowing the business to prioritize high-margin inventory over high-volume, low-profit items.

**`Performance Benchmarking:`** Provides a clear comparison of individual product success across different categories, revealing which specific models or brands drive the most financial growth.

**`Strategic Inventory Insight:`** The "Emerald Green" color scale visually separates the top-tier earners from average performers, making it easy for stakeholders to spot the products that should lead future marketing campaigns.

---

### Shipping Impact

In [89]:
# --- Shipping Cost Impact Analysis Table ---

# 1. Calculating summary statistics for Shipping Cost by Region
shipping_summary = df.groupby('Region')['Shipping_Cost'].agg([
    'count', 
    'mean', 
    'median', 
    'max'
]).sort_values(by='mean', ascending=False)

# 2. Renaming columns for a professional look
shipping_summary.columns = ['Total Orders', 'Avg Shipping Cost', 'Median Cost', 'Max Cost']

# 3. Applying 'styling' to make it look like a professional dashboard table
styled_table = shipping_summary.style.background_gradient(cmap='Reds', subset=['Avg Shipping Cost'])\
                                     .format("${:.2f}", subset=['Avg Shipping Cost', 'Median Cost', 'Max Cost'])\
                                     .set_caption("<b>Table 1: Regional Logistics Cost Breakdown</b>")

styled_table

,Total Orders,Avg Shipping Cost,Median Cost,Max Cost
Region,,,,
Middle East & Africa,208,$16.00,$14.82,$37.80
South America,191,$15.44,$13.66,$40.44
Asia Pacific,520,$14.48,$12.80,$39.83
Europe,503,$12.30,$10.99,$31.89
North America,578,$10.05,$8.75,$33.58


### Analysis:
While the table above confirms that and `Middle East & Africa`(MEA) and `South America` have the highest average logistics overhead, the following Boxplot allows us to see the distribution of these costs and identify specific high-cost outliers that are draining our profit margins.

In [90]:
# Create a Box Plot to see Shipping Cost distribution and Outliers by Region
fig_ship = px.box(
    df, 
    x='Region', 
    y='Shipping_Cost', 
    color='Region',
    points="outliers",  # This explicitly shows the "expensive" individual orders
    title="<b>📦 Shipping Cost Outliers by Region</b>",
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Safe
)

fig_ship.update_layout(
    title_x=0.5,
    yaxis_title="Shipping Cost ($)",
    showlegend=False
)

fig_ship.show()

#### Analysis: Shipping Cost Distribution

**`The Box:`** Represents the "normal" range of shipping costs (the middle 50% of orders).

**`The Dots (Outliers):`** Represent specific orders where shipping was unusually expensive.

**`The Insight:`** We can see which regions suffer from "unpredictable" logistics, helping us identify where to negotiate better carrier rates.

---

### Customer Segment Distribution

In [ ]:
# --- Customer Segment Volume Table ---

# Create the loyalty_data summary first
loyalty_data = df.groupby('Customer_Segment')['Quantity'].sum().reset_index()

# Now your existing code will work:
total_volume = loyalty_data['Quantity'].sum()
loyalty_data['Share %'] = (loyalty_data['Quantity'] / total_volume) * 100

# 2. Sort by highest volume
loyalty_data = loyalty_data.sort_values(by='Quantity', ascending=False)

# 3. Create a styled table
styled_loyalty = loyalty_data.style.background_gradient(cmap='GnBu', subset=['Quantity'])\
                                   .format("{:.1f}%", subset=['Share %'])\
                                   .format("{:,}", subset=['Quantity'])\
                                   .set_caption("<b>Table 2: Customer Segment Order Distribution</b>")

styled_loyalty

,Customer_Segment,Quantity,Share %
0,Consumer,"3,602",50.6%
1,Corporate,"2,195",30.9%
2,Home Office,"1,318",18.5%


In [92]:
# Summing Quantity by Segment
loyalty_data = df.groupby('Customer_Segment')['Quantity'].sum().reset_index()

fig_loyalty = px.pie(
    loyalty_data, 
    values='Quantity', 
    names='Customer_Segment', 
    hole=0.5, # Makes it a donut
    title="<b>👥 Order Volume by Customer Segment</b>",
    color_discrete_sequence=px.colors.sequential.Tealgrn
)

# Pull the largest segment out slightly for style
fig_loyalty.update_traces(textinfo='percent+label', pull=[0.1, 0, 0])
fig_loyalty.update_layout(title_x=0.5)
fig_loyalty.show()

### Profit Efficiency Analysis

In [93]:
# --- Segment Profitability Efficiency ---

# 1. Calculate Profit per Unit for each segment
segment_efficiency = df.groupby('Customer_Segment').agg({
    'Quantity': 'sum',
    'Profit': 'sum'
}).reset_index()

segment_efficiency['Profit_Per_Unit'] = segment_efficiency['Profit'] / segment_efficiency['Quantity']

# 2. Visualize the Efficiency
fig_efficiency = px.bar(
    segment_efficiency.sort_values('Profit_Per_Unit', ascending=False),
    x='Customer_Segment',
    y='Profit_Per_Unit',
    text_auto='.2f',
    title="<b>💰 Profit Efficiency: Which Segment is Most Profitable?</b>",
    labels={'Profit_Per_Unit': 'Profit per Unit Sold ($)'},
    color='Profit_Per_Unit',
    color_continuous_scale='Greens',
    range_color=[0, 45],                     # Anchoring at 0 and giving some buffer above 24.24
    template='plotly_white'
)

fig_efficiency.update_layout(title_x=0.5)
fig_efficiency.show()

### 📊 Analysis: Customer Segment Performance

**`Market Penetration (Volume):`** As shown in the Order Distribution Table, the Consumer segment is our high-volume engine. This confirms that our primary brand strength lies in individual retail sales, moving the largest quantity of units across all regions.

**`The Efficiency Leader:`** Surprisingly, the Profit Efficiency Chart reveals that the Consumer segment is also our most profitable per unit ($24.24), outperforming Corporate and Home Office by nearly 20%.

**`Strategic Insight:`** Unlike many businesses where bulk corporate orders are most profitable, our retail Consumer base offers the best "bang for our buck." We should prioritize marketing spend and loyalty programs for Consumers, as they provide both the highest volume and the best profit margins.

---
---

## 🏁 Final Project Executive Summary

**`1. The Goal`**
The objective of this analysis was to evaluate global sales performance and identify specific areas—both geographical and behavioral—where profit was being lost or gained.

**`2. Key Discoveries`**
The Profit "Suspects": While South America and MEA showed the lowest net profit, our diagnostic phase revealed this was driven by high logistics overhead (shipping costs ~50% higher than North America) rather than low sales.

**`The Power Segment:`** The Consumer segment is the business's greatest asset. It leads in both total volume and Profit Efficiency, yielding $24.24 per unit (20% higher than Corporate/Home Office).

### Product Performance: 
Top-tier profitable products were identified, providing a roadmap for future inventory focus.

**`3. Strategic Recommendations`**
Logistics: Implement a "Minimum Order Value" for free shipping in South America and MEA to protect margins from high shipping costs.

**`Marketing:`** Double down on Consumer-focused campaigns. Since this segment has the highest efficiency, every marketing dollar spent here generates the best return.

**`Optimization:`** Review the "Shipping Impact" for low-margin products in high-cost regions to decide if certain products should be "Region-Locked."

---

## 💾 Exporting Visuals for Documentation

In [94]:
# 2. Save your figures (Variable names updated to match your notebook)
# Note: 'fig' is your Regional Sales vs Profit bar chart
fig.write_image("./images/regional_performance.png", scale=2)

# Save the Sunburst Chart
fig_sun.write_image("./images/sales_hierarchy.png", scale=2)

# Save the Trends Chart
fig_trend.write_image("./images/trends_lines.png", scale=2)

# Save the Top Products Chart
fig_map.write_image("./images/top_products.png", scale=2)

# Save the Top Products Chart
fig_prod.write_image("./images/top_products.png", scale=2)

# Save the Sales Momentum Chart
fig_ship.write_image("./images/sales_momentum.png", scale=2)

# Save the Customer Segment Donut Chart
fig_loyalty.write_image("./images/customer_segments.png", scale=2)

# Save the new Efficiency Chart
fig_efficiency.write_image("./images/segment_efficiency.png", scale=2)

print("✅ All plots saved successfully in the 'images' folder!")

✅ All plots saved successfully in the 'images' folder!
